# Data Preprocessing & Feature Engineering - Patient Health Records




Import Libraries

In [8]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer


from sklearn.impute import IterativeImputer

from scipy import stats
from scipy.stats.mstats import winsorize




## Part A: Handling Missing Values

### Task 1: Identify missing values and create a summary report

In [5]:
df = pd.read_csv("patient_health_records_raw.csv")

In [6]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary


,Missing Count,Missing %
patient_id,0,0.0
age,30,6.0
gender,25,5.0
region,35,7.0
bmi,30,6.0
blood_pressure,0,0.0
cholesterol,30,6.0
glucose,30,6.0
disease_risk,0,0.0


From the table above we can see that `age`, `gender`, `region`, `bmi`, `cholesterol` and `glucose`
have missing values, while `patient_id`, `blood_pressure` and `disease_risk` have none.

### Task 2: Apply different imputation techniques and compare results

We will keep the original dirty dataframe (`df`) untouched, and try each technique on a **copy**,
so we can clearly compare the results of each method.

**2.1 Simple Imputer (Numerical) - fill missing BMI with mean**

In [ ]:
df_bmi = df.copy()

mean_imputer = SimpleImputer(strategy='mean')
df_bmi['bmi'] = mean_imputer.fit_transform(df_bmi[['bmi']])

print("Missing values in bmi after mean imputation:", df_bmi['bmi'].isnull().sum())
df_bmi[['bmi']].describe()


Missing values in bmi after mean imputation: 0


,bmi
count,500.000000
mean,26.687447
std,5.154758
min,8.000000
25%,23.800000
50%,26.687447
75%,28.800000
max,65.000000


**2.2 Simple Imputer (Categorical) - fill missing Region with most frequent value**

In [ ]:
df_region = df.copy()

region_imputer = SimpleImputer(strategy='most_frequent')
df_region['region'] = region_imputer.fit_transform(df_region[['region']]).ravel()


print("Missing values in region after imputation:", df_region['region'].isnull().sum())
df_region['region'].value_counts()


Missing values in region after imputation: 0


region
East     161
North    118
South    118
West     103
Name: count, dtype: int64

**2.3 Most Frequent Imputation - fill missing Gender with most common category**

In [ ]:
df_gender = df.copy()

gender_imputer = SimpleImputer(strategy='most_frequent')
df_gender['gender'] = gender_imputer.fit_transform(df_gender[['gender']]).ravel()

print("Missing values in gender after imputation:", df_gender['gender'].isnull().sum())
df_gender['gender'].value_counts()


Missing values in gender after imputation: 0


gender
Male      272
Female    228
Name: count, dtype: int64

**2.4 Missing Indicator + Random Sample Imputation - for Age**

Here we do two things:
1. Create a new binary column that tells us whether the value was originally missing (1) or not (0).
2. Fill the missing values by randomly picking values from the ones that are already present in the column.

In [ ]:
df_age = df.copy()


df_age['age_was_missing'] = df_age['age'].isnull().astype(int)

known_ages = df_age['age'].dropna().values
missing_mask = df_age['age'].isnull()
df_age.loc[missing_mask, 'age'] = np.random.choice(known_ages, missing_mask.sum())

print("Missing values in age after imputation:", df_age['age'].isnull().sum())
df_age[['age', 'age_was_missing']].head()


Missing values in age after imputation: 0


,age,age_was_missing
0,69.0,0
1,32.0,0
2,89.0,0
3,78.0,0
4,32.0,1


**2.5 KNN Imputer - multivariate imputation**

KNN Imputer looks at the other numerical columns of a patient (its "neighbors") and uses their
values to guess the missing value. It works on multiple numeric columns together.

In [ ]:
df_knn = df.copy()
numeric_cols = ['age', 'bmi', 'cholesterol', 'glucose']

knn_imputer = KNNImputer(n_neighbors=5)
df_knn[numeric_cols] = knn_imputer.fit_transform(df_knn[numeric_cols])

print("Missing values after KNN imputation:")
df_knn[numeric_cols].isnull().sum()


Missing values after KNN imputation:


age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64

**2.6 MICE Algorithm (chained equations)**

MICE = Multivariate Imputation by Chained Equations. It imputes each column by building a small
regression model using the other columns, and repeats this process multiple times until the
values stabilize. In sklearn this is called `IterativeImputer`.

In [ ]:
df_mice = df.copy()


mice_imputer = IterativeImputer(random_state=42, max_iter=10)
df_mice[numeric_cols] = mice_imputer.fit_transform(df_mice[numeric_cols])

print("Missing values after MICE imputation:")
df_mice[numeric_cols].isnull().sum()


Missing values after MICE imputation:


age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64

### Comparing the imputation results

Below we compare the mean of `bmi` before and after a couple of the techniques above, just to see
how much each method changes the overall distribution of the column.

In [ ]:
comparison = pd.DataFrame({
    'Original (ignoring NaN)': df[numeric_cols].mean(),
    'After KNN Imputer': df_knn[numeric_cols].mean(),
    'After MICE': df_mice[numeric_cols].mean()
})

comparison.round(2)


,Original (ignoring NaN),After KNN Imputer,After MICE
age,52.62,52.79,52.61
bmi,26.69,26.70,26.69
cholesterol,199.87,199.91,199.89
glucose,100.86,100.77,100.86


**Building the final imputed dataset**

For the rest of the project we need one single clean dataframe. We choose MICE for the numeric
columns (since it usually gives the most realistic values by considering relationships between
columns), and most frequent imputation for the categorical columns.

In [ ]:
df_imputed = df.copy()

df_imputed[numeric_cols] = df_mice[numeric_cols]


df_imputed['gender'] = SimpleImputer(strategy='most_frequent').fit_transform(df_imputed[['gender']]).ravel()
df_imputed['region'] = SimpleImputer(strategy='most_frequent').fit_transform(df_imputed[['region']]).ravel()

print("Total missing values left:", df_imputed.isnull().sum().sum())
df_imputed.head()


Total missing values left: 0


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P1000,69.00000,Female,East,27.0,140.1,168.400000,100.3,1
1,P1001,32.00000,Female,East,29.9,113.4,198.473338,80.9,0
2,P1002,89.00000,Male,West,32.7,126.8,228.500000,91.9,1
3,P1003,78.00000,Male,North,30.1,139.5,201.783520,113.7,1
4,P1004,52.56634,Male,East,18.6,190.0,196.900000,102.1,0


## Part B: Handling Outliers

Now that missing values are handled, we work with `df_imputed` and detect/treat the outliers in
`bmi`, `blood_pressure`, `cholesterol` and `glucose`.

**3.1 Z-score method - detect patients with extreme cholesterol / glucose values**

In [ ]:
z_cholesterol = np.abs(stats.zscore(df_imputed['cholesterol']))
z_glucose = np.abs(stats.zscore(df_imputed['glucose']))

z_outliers = df_imputed[(z_cholesterol > 3) | (z_glucose > 3)]
print("Number of outliers found using Z-score method:", len(z_outliers))

# removing these outliers to see the effect
df_after_zscore = df_imputed[(z_cholesterol <= 3) & (z_glucose <= 3)]
print("Shape before removal:", df_imputed.shape)
print("Shape after Z-score removal:", df_after_zscore.shape)


Number of outliers found using Z-score method: 9
Shape before removal: (500, 9)
Shape after Z-score removal: (491, 9)


**3.2 IQR method - detect unusual BMI values**

In [ ]:
Q1 = df_imputed['bmi'].quantile(0.25)
Q3 = df_imputed['bmi'].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

iqr_outliers = df_imputed[(df_imputed['bmi'] < lower_limit) | (df_imputed['bmi'] > upper_limit)]
print("Number of outliers found using IQR method:", len(iqr_outliers))

df_after_iqr = df_imputed[(df_imputed['bmi'] >= lower_limit) & (df_imputed['bmi'] <= upper_limit)]
print("Shape before removal:", df_imputed.shape)
print("Shape after IQR removal:", df_after_iqr.shape)


Number of outliers found using IQR method: 9
Shape before removal: (500, 9)
Shape after IQR removal: (491, 9)


**3.3 Percentile method - cap blood pressure below 1st percentile and above 99th percentile**

In [ ]:
low_limit = df_imputed['blood_pressure'].quantile(0.01)
high_limit = df_imputed['blood_pressure'].quantile(0.99)

df_percentile = df_imputed.copy()
df_percentile['blood_pressure'] = df_percentile['blood_pressure'].clip(lower=low_limit, upper=high_limit)

print("Blood pressure before capping:")
print(df_imputed['blood_pressure'].describe())
print("\nBlood pressure after percentile capping:")
print(df_percentile['blood_pressure'].describe())


Blood pressure before capping:
count    500.000000
mean     122.119200
std       14.945971
min       84.900000
25%      113.000000
50%      120.850000
75%      129.125000
max      210.000000
Name: blood_pressure, dtype: float64

Blood pressure after percentile capping:
count    500.000000
mean     122.095930
std       14.552705
min       91.593000
25%      113.000000
50%      120.850000
75%      129.125000
max      190.000000
Name: blood_pressure, dtype: float64


**3.4 Winsorization - cap extreme outliers instead of removing them**

Winsorization is similar to the percentile method, but instead of doing it manually, we use
`scipy`'s `winsorize` function. Here we apply it to all 4 columns that had outliers.
We use this as our **final outlier treatment method**, because removing rows (like in Z-score /
IQR above) throws away data, while capping keeps every patient's record in the dataset.

In [ ]:
df_final = df_imputed.copy()

outlier_cols = ['bmi', 'blood_pressure', 'cholesterol', 'glucose']


for col in outlier_cols:
    df_final[col] = winsorize(df_final[col], limits=[0.01, 0.01])

print("Winsorization applied on:", outlier_cols)
df_final[outlier_cols].describe()


Winsorization applied on: ['bmi', 'blood_pressure', 'cholesterol', 'glucose']


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


,bmi,blood_pressure,cholesterol,glucose
count,500.000000,500.000000,500.000000,500.000000
mean,26.502902,122.096000,199.734088,100.976281
std,3.943455,14.552558,29.021620,18.910380
min,17.200000,91.600000,123.100000,55.600000
25%,23.800000,113.000000,181.575000,87.650000
50%,26.673492,120.850000,199.675078,100.874667
75%,28.800000,129.125000,218.200000,113.450000
max,36.300000,190.000000,272.400000,147.300000


**Task 5: Compare dataset shape and summary before vs after outlier treatment**

In [ ]:
print("Shape before outlier treatment:", df_imputed.shape)
print("Shape after outlier treatment (winsorization):", df_final.shape)

print("\nSummary BEFORE outlier treatment:")
print(df_imputed[outlier_cols].describe().round(2))

print("\nSummary AFTER outlier treatment:")
print(df_final[outlier_cols].describe().round(2))


Shape before outlier treatment: (500, 9)
Shape after outlier treatment (winsorization): (500, 9)

Summary BEFORE outlier treatment:
          bmi  blood_pressure  cholesterol  glucose
count  500.00          500.00       500.00   500.00
mean    26.69          122.12       199.89   100.86
std      5.15           14.95        34.49    19.78
min      8.00           84.90        35.00    25.00
25%     23.80          113.00       181.58    87.65
50%     26.67          120.85       199.68   100.87
75%     28.80          129.12       218.20   113.45
max     65.00          210.00       450.00   162.20

Summary AFTER outlier treatment:


          bmi  blood_pressure  cholesterol  glucose
count  500.00          500.00       500.00   500.00
mean    26.50          122.10       199.73   100.98
std      3.94           14.55        29.02    18.91
min     17.20           91.60       123.10    55.60
25%     23.80          113.00       181.58    87.65
50%     26.67          120.85       199.68   100.87
75%     28.80          129.12       218.20   113.45
max     36.30          190.00       272.40   147.30


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


## Part C: Final Clean Dataset

`df_final` now has:
- No missing values (handled using MICE + most frequent imputation)
- No extreme outliers (handled using winsorization)

This is our machine-learning-ready dataset.

In [ ]:
print("Missing values in final dataset:", df_final.isnull().sum().sum())
print("Final dataset shape:", df_final.shape)
df_final.head(10)


Missing values in final dataset: 0
Final dataset shape: (500, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P1000,69.00000,Female,East,27.0,140.1,168.400000,100.3,1
1,P1001,32.00000,Female,East,29.9,113.4,198.473338,80.9,0
2,P1002,89.00000,Male,West,32.7,126.8,228.500000,91.9,1
3,P1003,78.00000,Male,North,30.1,139.5,201.783520,113.7,1
4,P1004,52.56634,Male,East,18.6,190.0,196.900000,102.1,0
5,P1005,41.00000,Female,East,20.9,117.6,194.900000,111.7,1
6,P1006,20.00000,Female,North,23.5,113.0,202.100000,139.5,0
7,P1007,39.00000,Male,South,26.1,107.8,234.900000,68.7,0
8,P1008,70.00000,Male,South,28.1,112.2,172.200000,132.3,0
9,P1009,19.00000,Female,West,23.1,105.3,207.200000,102.1,0


In [ ]:
# saving the final cleaned dataset
df_final.to_csv('patient_health_records_cleaned.csv', index=False)
print("Cleaned dataset saved as patient_health_records_cleaned.csv")


Cleaned dataset saved as patient_health_records_cleaned.csv


### Report

**Which imputation strategy was most effective?**
MICE (IterativeImputer) worked best for the numeric columns (`age`, `bmi`, `cholesterol`,
`glucose`) because it looks at the relationship between all the numeric columns together while
filling missing values, instead of just using one fixed number like the mean. KNN Imputer gave
similar results but MICE was chosen since it is generally more accurate for datasets with
multiple correlated numeric columns. For the categorical columns (`gender`, `region`), simple
"most frequent" imputation was good enough since these columns only have a few categories.

**Which outlier handling method preserved data quality best?**
Winsorization preserved data quality the best. The Z-score and IQR methods both worked correctly
for detecting outliers, but they removed the outlier rows completely, which means we lose
patient records (and possibly useful information about the other columns of that patient).
Winsorization instead caps the extreme values to a reasonable range (1st/99th percentile), so we
keep all 500 patient records while still removing the effect of unrealistic extreme values.

**How did data cleaning improve dataset usability?**
- Before cleaning, the dataset had missing values in 6 columns and extreme/unrealistic values in
  4 columns, which would have caused errors or wrong predictions if used directly in a machine
  learning model.
- After cleaning, every column has complete data and the numeric columns are within a realistic
  range, which makes the dataset ready to be used for training a heart disease risk prediction
  model.
